# RHIZO-NET: Notebook 07 - Complete End-to-End Pipeline Demo
Runs the complete RHIZO-NET pipeline from raw image -> segmentation -> topology -> fusion -> TNAU recommendation.

In [ ]:
import numpy as np
import torch
from src.unet.rhizo_attention_net import RhizoAttentionNet
from src.topology.skeletonize import skeletonize_root_mask
from src.topology.graph_extract import extract_root_graph
from src.topology.phenotype_features import extract_phenotype_features
from src.fusion.soil_features import process_soilgrids_features, build_numerical_vector
from src.agronomic.recommendation_engine import TNAUAgronomicEngine
from src.utils.soilgrids_client import fetch_soilgrids_data
from src.utils.visualization import plot_pipeline_summary

print('='*60)
print('RHIZO-NET COMPLETE PIPELINE DEMONSTRATION')
print('='*60)

# 1. Load sample image (dummy synthetic)
dummy_image = np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
dummy_mask = np.zeros((256, 256), dtype=np.uint8)
dummy_mask[30:220, 128] = 255
dummy_mask[80:150, 128:190] = 255

# 2. Topology
skel = skeletonize_root_mask(dummy_mask)
graph, meta = extract_root_graph(skel)
pheno = extract_phenotype_features(graph)

# 3. SoilGrids & Agronomic Engine
raw_soil = fetch_soilgrids_data(lat=11.0168, lon=76.9558) # Coimbatore, TN
engine = TNAUAgronomicEngine()
recommendation = engine.generate_recommendation(
    deficiency_class='nitrogen_deficiency',
    crop_name='sorghum_irrigated',
    soil_n_kg_ha=190.0,
    soil_p_kg_ha=14.0,
    soil_k_kg_ha=220.0,
    soil_ph=7.2,
)

print('✓ Pipeline execution complete! Displaying summary report...')
plot_pipeline_summary(dummy_image, dummy_mask, graph, recommendation)
